# Day 1 — Train / Validation / Test Splits

### Step 1 - Train/Validation/Test Split

In [129]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier

In [130]:
Bank_data = pd.read_csv("Bank_Customer_Churn_Prediction.csv")
Bank_data

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [131]:
Bank_data = pd.get_dummies(Bank_data,columns=['country'], drop_first=True)
Bank_data['gender'] = Bank_data['gender'].map({'Male': 0, 'Female': 1})

In [132]:
x = Bank_data.drop(["churn"], axis = 1)
y = Bank_data["churn"]

### Step 2 - Training & Tuning the Model

In [133]:
X_temp, X_test, y_temp, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [134]:
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

In [135]:
len(X_train), len(X_val), len(X_test)

(6000, 2000, 2000)

#### The dataset is trained using  Random Forest Classifier 

In [128]:
clf = RandomForestClassifier()
clf.fit(X_train, y_train)
y_preds=clf.predict(X_val)
clf.get_params()

print("Baseline Accuracy:", accuracy_score(y_val, y_preds))
print("Baseline F1:", f1_score(y_val, y_preds))  
print("Baseline Precision:", precision_score(y_val, y_preds))      
print("Baseline Recall:", recall_score(y_val, y_preds))

Baseline Accuracy: 0.864
Baseline F1: 0.5903614457831325
Baseline Precision: 0.7808764940239044
Baseline Recall: 0.4745762711864407


In [126]:
## second classifier - n_estimators adjusted
from sklearn.metrics import accuracy_score, f1_score, precision_score , recall_score
depths = [15,18,20,22]


for d in depths:
    clf_2 = RandomForestClassifier(max_depth=d, random_state=42)
    clf_2.fit(X_train, y_train)
    y_preds=clf_2.predict(X_val)
    print(f"Accuracy: {accuracy_score(y_val, y_preds):.4f}")
    print(f"F1: {f1_score(y_val, y_preds):.4f}")  
    print(f"Precision: {precision_score(y_val, y_preds):.4f}")      
    print(f"Recall: {recall_score(y_val, y_preds):.4f}")
    print("===========")

Accuracy: 0.8645
F1: 0.5837
Precision: 0.7983
Recall: 0.4600
Accuracy: 0.8670
F1: 0.5982
Precision: 0.7952
Recall: 0.4794
Accuracy: 0.8695
F1: 0.6110
Precision: 0.7946
Recall: 0.4964
Accuracy: 0.8645
F1: 0.5973
Precision: 0.7731
Recall: 0.4867


    The data was trained on random_set = 42 & and tuned by changing the max_depth parameter
    The data was split into 60/20/20 - train/validation/test
    The best tuned results were recorded when max_depth  = 20 and were used later in testing.

    Tuning gave a good improvement, exceeding the baseline slightly

### Step 3 - Final Model Evaluation on the Test Set

In [127]:
clf_test = RandomForestClassifier(max_depth=20, random_state=42)
clf_test.fit(X_train, y_train)

test_predictions = clf_test.predict(X_test)
print(f"Final test accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print(f"Final test F1: {f1_score(y_test, test_predictions):.4f}")  
print(f"Final test Precision: {precision_score(y_test, test_predictions):.4f}")      
print(f"Final test Recall: {recall_score(y_test, test_predictions):.4f}")

Final test accuracy: 0.8630
Final test F1: 0.5732
Final test Precision: 0.7390
Final test Recall: 0.4682


    The final model results show a a slight decrease in accuracy but notable reductions  in F1, Precision and Recall a bit below the baseline. This drop  is small and does not indicate overfitting but the tuning didn't carry over to test.

    Recall remained low around 0.46 accross results of the 3 phases. That means the model misses more than half of the churners when predicting. In comparison to the later, Precision remains high accross models- those who actually churned that the model was able to detect.

### Step 4 - Tuning Against the Test Set Consequences

    If the the Model was tuned against the test set that will lead to data leakage; the model memorizing the data rather than learning the patterns and predicting results based on it. As a result the model will show perfect performance while training and tuning but once it is tested using real world, new data, it will show bias and a huge reduction in performance whic makes it useless in a real world setting. 